In [ ]:
# FINAL – 3 REAL PHONES (Nothing_2a, Poco-M3, Pixel) – NO ERRORS
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json, numpy as np, tensorflow as tf
from pathlib import Path
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import applications, layers, models, callbacks

# SETTINGS
MERGED = "/content/merged_mobile"
RES    = "/content/drive/MyDrive/Merged_Model/results_mobile_only"
CKPT   = f"{RES}/checkpoints"
for p in [RES, CKPT]: Path(p).mkdir(parents=True, exist_ok=True)

# LINK ONLY 3 PHONES
!rm -rf {MERGED}
os.makedirs(f"{MERGED}/orig", exist_ok=True)
os.makedirs(f"{MERGED}/recap", exist_ok=True)

def link(src, dst, pref=""):
    cnt = 0
    src_path = Path(src)
    if not src_path.exists(): return 0
    for f in src_path.iterdir():
        if f.suffix.lower() in {".jpg",".jpeg",".png",".heic"}:
            name = f"{pref}_{f.name}" if pref else f.name
            (Path(dst)/name).symlink_to(f)
            cnt += 1
    return cnt

print("Linking 3 phones...")
for dev in ["Nothing_2a", "Poco-M3", "Pixel"]:
    link(f"/content/drive/MyDrive/{dev}/originals",  f"{MERGED}/orig",  dev.lower())
    link(f"/content/drive/MyDrive/{dev}/recaptures", f"{MERGED}/recap", dev.lower())

# SPLIT
orig_files  = list(Path(f"{MERGED}/orig").glob("*"))
recap_files = list(Path(f"{MERGED}/recap").glob("*"))
np.random.seed(42)
np.random.shuffle(orig_files); np.random.shuffle(recap_files)

train_o, val_o = orig_files[:int(0.8*len(orig_files))],  orig_files[int(0.8*len(orig_files)):]
train_r, val_r = recap_files[:int(0.8*len(recap_files))], recap_files[int(0.8*len(recap_files)):]

print(f"Train → Orig {len(train_o)} | Recap {len(train_r)}")
print(f"Val   → Orig {len(val_o)} | Recap {len(val_r)}")

# PREPROCESS + AUGMENT
def decode_img(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [224, 224])
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_jpeg_quality(img, 70, 100)
    return img

@tf.function
def process(path, label):
    img = decode_img(path)
    if tf.random.uniform(()) < 0.8:
        img = augment(img)
    return img, label

# DATASETS
train_paths = [str(p) for p in train_o + train_r]
train_labels = [0]*len(train_o) + [1]*len(train_r)

val_paths = [str(p) for p in val_o + val_r]
val_labels = [0]*len(val_o) + [1]*len(val_r)

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
val_ds   = tf.data.Dataset.from_tensor_slices((val_paths,   val_labels))

# Apply processing
train_ds = train_ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda p,l: (decode_img(p), l), num_parallel_calls=tf.data.AUTOTUNE)

# Oversample originals ×4
orig_train = train_ds.filter(lambda x,y: y == 0).repeat(4).shuffle(4000)
recap_train = train_ds.filter(lambda x,y: y == 1)
train_ds = orig_train.concatenate(recap_train).shuffle(8000).batch(32).prefetch(2)
val_ds   = val_ds.batch(32).prefetch(2)

# MODEL
inp = tf.keras.Input((224,224,3))
base = applications.EfficientNetB0(include_top=False, weights='imagenet')(inp)
x = layers.GlobalAveragePooling2D()(base)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
out = layers.Dense(1, activation='sigmoid')(x)
model = models.Model(inp, out)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# CALLBACKS
ckpt_path = f"{CKPT}/best.weights.h5"
cp = callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, save_weights_only=True,
                              monitor='val_accuracy', mode='max', verbose=1)
es = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1)

if os.path.exists(ckpt_path):
    print("Loading best checkpoint...")
    model.load_weights(ckpt_path)

# TRAIN
model.fit(train_ds, validation_data=val_ds, epochs=60, callbacks=[cp, es], verbose=1)

# EVALUATE & SAVE
y_true, y_pred = [], []
for x, y in val_ds:
    p = (model.predict(x, verbose=0) > 0.5).astype(int).flatten()
    y_true.extend(y.numpy())
    y_pred.extend(p)

cm = confusion_matrix(y_true, y_pred)
rep = classification_report(y_true, y_pred, target_names=['Original','Recaptured'], output_dict=True)
acc = cm.diagonal().sum() / cm.sum()

# SAVE EVERYTHING (2 decimals)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Original','Recaptured'],
            yticklabels=['Original','Recaptured'])
plt.title(f'Acc: {acc:.2%} | F1-O: {rep["Original"]["f1-score"]:.2f} | F1-R: {rep["Recaptured"]["f1-score"]:.2f}')
plt.savefig(f"{RES}/confusion_matrix.png", dpi=200, bbox_inches='tight'); plt.close()

pd.DataFrame(rep).transpose().round(2).to_csv(f"{RES}/report.csv")

summary = {
    "accuracy": round(acc, 4),
    "f1_original": round(rep["Original"]["f1-score"], 4),
    "f1_recaptured": round(rep["Recaptured"]["f1-score"], 4)
}
with open(f"{RES}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

model.save(f"{RES}/mobile_model.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
open(f"{RES}/mobile_detector.tflite", "wb").write(tflite_model)

print("\nDONE! Results (real phones only):")
print(f"Accuracy : {acc:.2%}")
print(f"F1 Original   : {rep['Original']['f1-score']:.3f}")
print(f"F1 Recaptured : {rep['Recaptured']['f1-score']:.3f}")
print("All files saved →", RES)

Mounted at /content/drive
Linking 3 phones...
Train → Orig 271 | Recap 267
Val   → Orig 68 | Recap 67
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/60
     43/Unknown 702s 1s/step - accuracy: 0.7714 - loss: 0.5779   

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_accuracy improved from -inf to 0.62963, saving model to /content/drive/MyDrive/Merged_Model/results_mobile_only/checkpoints/best.weights.h5
43/43 ━━━━━━━━━━━━━━━━━━━━ 794s 3s/step - accuracy: 0.7716 - loss: 0.5772 - val_accuracy: 0.6296 - val_loss: 0.8667
Epoch 2/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.8064 - loss: 0.4939
Epoch 2: val_accuracy improved from 0.62963 to 0.81481, saving model to /content/drive/MyDrive/Merged_Model/results_mobile_only/checkpoints/best.weights.h5
43/43 ━━━━━━━━━━━━━━━━━━━━ 345s 701ms/step - accuracy: 0.8066 - loss: 0.4938 - val_accuracy: 0.8148 - val_loss: 0.3892
Epoch 3/60
42/43 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.8200 - loss: 0.4925
Epoch 3: val_accuracy did not improve from 0.81481
43/43 ━━━━━━━━━━━━━━━━━━━━ 374s 611ms/step - accuracy: 0.8192 - loss: 0.4933 - val_accuracy: 0.7185 - val_loss: 0.5563
Epoch 4/60
42/43 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.8021 - loss: 0.4976
Epoch 4: val_accuracy did n